# Weeks 4-5 Residential Data Cleaning and California Preparation

**IDX Exchange Data Analyst Internship**  
**Analysis period:** January 2024 through June 2026

This report documents the cleaning decisions applied after the Week 2
validation and Week 3 market analysis. It covers new June rows, removed
columns, invalid numeric values, date inconsistencies, and the final
California-only Residential datasets.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import HTML, display


ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

CLEANED_DIR = ROOT / "data" / "cleaned"
ENRICHED_DIR = ROOT / "data" / "enriched"
PROCESSED_DIR = ROOT / "data" / "processed"

COLORS = {
    "ink": "#1F2937",
    "teal": "#0F766E",
    "blue": "#1F6B8C",
    "amber": "#D97706",
    "red": "#B91C1C",
    "green": "#15803D",
    "gray": "#6B7280",
}

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

In [2]:
cleaning = pd.read_csv(CLEANED_DIR / "cleaning_summary.csv")
column_removals = pd.read_csv(CLEANED_DIR / "column_removal_report.csv")
data_types = pd.read_csv(CLEANED_DIR / "data_type_report.csv")
listing_months = pd.read_csv(
    PROCESSED_DIR / "combined_listings_row_counts.csv"
)
sold_months = pd.read_csv(PROCESSED_DIR / "combined_sold_row_counts.csv")

required_reports = {
    "cleaning summary": cleaning,
    "column removals": column_removals,
    "data types": data_types,
    "listing monthly counts": listing_months,
    "sold monthly counts": sold_months,
}
assert all(not report.empty for report in required_reports.values())


def cleaning_value(dataset, metric):
    match = cleaning[
        cleaning["dataset"].eq(dataset)
        & cleaning["metric"].eq(metric)
    ]
    if match.empty:
        raise KeyError(f"Missing cleaning metric: {dataset}/{metric}")
    return int(match.iloc[0]["value"])


print("Weeks 4-5 cleaning report inputs loaded successfully.")

Weeks 4-5 cleaning report inputs loaded successfully.


## Final Cleaning Snapshot

In [3]:
final_listings = cleaning_value("listings_residential", "output_rows")
final_sold = cleaning_value("sold_residential", "output_rows")
removed_rows = (
    cleaning_value("listings_residential", "rows_removed_total")
    + cleaning_value("sold_residential", "rows_removed_total")
)
removed_columns = (
    cleaning_value("listings_residential", "columns_removed")
    + cleaning_value("sold_residential", "columns_removed")
)

dashboard_html = f'''
<div style="display:grid;grid-template-columns:repeat(4,1fr);gap:14px;margin:8px 0 18px 0;">
  <div style="border-top:5px solid {COLORS["teal"]};padding:16px;background:#F7FAFA;">
    <div style="color:{COLORS["gray"]};font-size:13px;">Final California Listings</div>
    <div style="color:{COLORS["ink"]};font-size:28px;font-weight:700;">{final_listings:,}</div>
  </div>
  <div style="border-top:5px solid {COLORS["blue"]};padding:16px;background:#F7F9FB;">
    <div style="color:{COLORS["gray"]};font-size:13px;">Final California Sold</div>
    <div style="color:{COLORS["ink"]};font-size:28px;font-weight:700;">{final_sold:,}</div>
  </div>
  <div style="border-top:5px solid {COLORS["amber"]};padding:16px;background:#FFFBF3;">
    <div style="color:{COLORS["gray"]};font-size:13px;">Rows Removed</div>
    <div style="color:{COLORS["ink"]};font-size:28px;font-weight:700;">{removed_rows:,}</div>
  </div>
  <div style="border-top:5px solid {COLORS["red"]};padding:16px;background:#FFF7F7;">
    <div style="color:{COLORS["gray"]};font-size:13px;">Columns Removed</div>
    <div style="color:{COLORS["ink"]};font-size:28px;font-weight:700;">{removed_columns:,}</div>
  </div>
</div>
'''
display(HTML(dashboard_html))

## June 2026 Data Addition

The June monthly files were added before cleaning. Only the Residential rows
continued into the Weeks 4-5 process.

In [4]:
june_rows = pd.concat(
    [
        listing_months[
            pd.to_numeric(listing_months["month"], errors="coerce").eq(202606)
        ],
        sold_months[
            pd.to_numeric(sold_months["month"], errors="coerce").eq(202606)
        ],
    ],
    ignore_index=True,
)[
    ["group", "file", "total_rows", "residential_rows", "non_residential_rows"]
].copy()
june_rows.columns = [
    "Dataset",
    "June Source File",
    "New Rows",
    "New Residential Rows",
    "New Non-Residential Rows",
]

display(
    june_rows.style
    .hide(axis="index")
    .format(
        {
            "New Rows": "{:,.0f}",
            "New Residential Rows": "{:,.0f}",
            "New Non-Residential Rows": "{:,.0f}",
        }
    )
)

Dataset,June Source File,New Rows,New Residential Rows,New Non-Residential Rows
listings,CRMLSListing202606.csv,"37,450","24,074","13,376"
sold,CRMLSSold202606.csv,"25,511","17,537","7,974"


## Week 4: Cleaning Decisions

The cleaning script converted the required date and numeric fields, removed
fields with more than 90% missing data, and flagged invalid numeric values.
Missing values were left blank instead of being filled with estimates.

In [5]:
cleaning_overview = pd.DataFrame(
    [
        {
            "Dataset": "Residential Listings",
            "Rows Before": cleaning_value("listings_residential", "source_rows"),
            "Rows After": cleaning_value("listings_residential", "output_rows"),
            "Rows Removed": cleaning_value("listings_residential", "rows_removed_total"),
            "Columns Before": cleaning_value("listings_residential", "source_columns"),
            "Columns Removed": cleaning_value("listings_residential", "columns_removed"),
            "Final Columns": cleaning_value("listings_residential", "output_columns"),
        },
        {
            "Dataset": "Residential Sold",
            "Rows Before": cleaning_value("sold_residential", "source_rows"),
            "Rows After": cleaning_value("sold_residential", "output_rows"),
            "Rows Removed": cleaning_value("sold_residential", "rows_removed_total"),
            "Columns Before": cleaning_value("sold_residential", "source_columns"),
            "Columns Removed": cleaning_value("sold_residential", "columns_removed"),
            "Final Columns": cleaning_value("sold_residential", "output_columns"),
        },
    ]
)
cleaning_overview["Rows Removed (%)"] = (
    cleaning_overview["Rows Removed"]
    / cleaning_overview["Rows Before"]
    * 100
)

display(
    cleaning_overview.style
    .hide(axis="index")
    .format(
        {
            "Rows Before": "{:,.0f}",
            "Rows After": "{:,.0f}",
            "Rows Removed": "{:,.0f}",
            "Columns Before": "{:,.0f}",
            "Columns Removed": "{:,.0f}",
            "Final Columns": "{:,.0f}",
            "Rows Removed (%)": "{:.3f}%",
        }
    )
)

Dataset,Rows Before,Rows After,Rows Removed,Columns Before,Columns Removed,Final Columns,Rows Removed (%)
Residential Listings,"616,072","615,679",393,75,13,73,0.064%
Residential Sold,"447,964","447,853",111,84,15,80,0.025%


### Invalid Numeric Values

Each invalid value was replaced with blank/missing while the property row was
retained. The flag columns identify affected records in the cleaned datasets.

In [6]:
numeric_rules = [
    ("ClosePrice", "Zero or below", "nonpositive", "close_price_nonpositive_flag"),
    ("LivingArea", "Zero or below", "nonpositive", "living_area_nonpositive_flag"),
    ("DaysOnMarket", "Below zero", "negative", "days_on_market_negative_flag"),
    ("BedroomsTotal", "Below zero", "negative", "bedrooms_negative_flag"),
    (
        "BathroomsTotalInteger",
        "Below zero",
        "negative",
        "bathrooms_negative_flag",
    ),
]

numeric_replacements = pd.DataFrame(
    [
        {
            "Field": field,
            "Invalid Rule": rule,
            "Listings Flagged": cleaning_value("listings_residential", flag),
            "Sold Flagged": cleaning_value("sold_residential", flag),
            "Replacement": "Blank/missing; row retained",
        }
        for field, rule, comparison, flag in numeric_rules
    ]
)

display(
    numeric_replacements.style
    .hide(axis="index")
    .format(
        {
            "Listings Flagged": "{:,.0f}",
            "Sold Flagged": "{:,.0f}",
        }
    )
)

Field,Invalid Rule,Listings Flagged,Sold Flagged,Replacement
ClosePrice,Zero or below,0,1,Blank/missing; row retained
LivingArea,Zero or below,393,165,Blank/missing; row retained
DaysOnMarket,Below zero,28,51,Blank/missing; row retained
BedroomsTotal,Below zero,0,0,Blank/missing; row retained
BathroomsTotalInteger,Below zero,0,0,Blank/missing; row retained


In [7]:
scripts_path = str(ROOT / "scripts")
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)

from data_cleaning_preparation import points_in_california


source_files = {
    "Residential Listings": ENRICHED_DIR / "listings_residential_with_mortgage_rates.csv",
    "Residential Sold": ENRICHED_DIR / "sold_residential_with_mortgage_rates.csv",
}
invalid_examples = []

for dataset_label, source_path in source_files.items():
    fields = [field for field, _, _, _ in numeric_rules]
    source = pd.read_csv(
        source_path,
        usecols=fields + ["StateOrProvince", "Latitude", "Longitude"],
        low_memory=False,
    )
    latitude = pd.to_numeric(source["Latitude"], errors="coerce")
    longitude = pd.to_numeric(source["Longitude"], errors="coerce")
    state = (
        source["StateOrProvince"]
        .astype("string")
        .str.strip()
        .str.upper()
        .fillna("")
    )
    has_coordinates = latitude.notna() & longitude.notna()
    california_rows = points_in_california(longitude, latitude) | (
        ~has_coordinates & state.eq("CA")
    )
    source = source.loc[california_rows]

    for field, rule, comparison, flag in numeric_rules:
        values = pd.to_numeric(source[field], errors="coerce")
        invalid = values.le(0) if comparison == "nonpositive" else values.lt(0)
        for invalid_value, occurrences in (
            values[invalid.fillna(False)].value_counts().head(8).items()
        ):
            invalid_examples.append(
                {
                    "Dataset": dataset_label,
                    "Field": field,
                    "Invalid Value": invalid_value,
                    "Occurrences": int(occurrences),
                    "Cleaned Value": "Blank/missing",
                }
            )

invalid_examples = pd.DataFrame(invalid_examples)
display(HTML("<h4>Invalid values actually found</h4>"))
display(
    invalid_examples.style
    .hide(axis="index")
    .format({"Invalid Value": "{:,.2f}", "Occurrences": "{:,.0f}"})
)

Dataset,Field,Invalid Value,Occurrences,Cleaned Value
Residential Listings,LivingArea,0.00,393,Blank/missing
Residential Listings,DaysOnMarket,-2.00,6,Blank/missing
Residential Listings,DaysOnMarket,-1.00,5,Blank/missing
Residential Listings,DaysOnMarket,-6.00,3,Blank/missing
Residential Listings,DaysOnMarket,-3.00,2,Blank/missing
Residential Listings,DaysOnMarket,-14.00,2,Blank/missing
Residential Listings,DaysOnMarket,-48.00,1,Blank/missing
Residential Listings,DaysOnMarket,-58.00,1,Blank/missing
Residential Listings,DaysOnMarket,-33.00,1,Blank/missing
Residential Sold,ClosePrice,0.00,1,Blank/missing


### Columns Removed

These are the only columns removed under the more-than-90%-missing rule.

In [8]:
removed_columns = column_removals.copy()
removed_columns.columns = [
    "Dataset",
    "Removed Column",
    "Reason",
    "Missing Rows",
    "Missing Percent",
]

display(
    removed_columns.style
    .hide(axis="index")
    .format({"Missing Rows": "{:,.0f}", "Missing Percent": "{:.2f}%"})
)

Dataset,Removed Column,Reason,Missing Rows,Missing Percent
listings_residential,FireplacesTotal,more than 90% missing,"616,072",100.00%
listings_residential,AboveGradeFinishedArea,more than 90% missing,"616,072",100.00%
listings_residential,TaxAnnualAmount,more than 90% missing,"616,072",100.00%
listings_residential,TaxYear,more than 90% missing,"616,072",100.00%
listings_residential,ElementarySchoolDistrict,more than 90% missing,"616,072",100.00%
listings_residential,BusinessType,more than 90% missing,"616,072",100.00%
listings_residential,CoveredSpaces,more than 90% missing,"616,072",100.00%
listings_residential,MiddleOrJuniorSchoolDistrict,more than 90% missing,"616,072",100.00%
listings_residential,BelowGradeFinishedArea,more than 90% missing,"612,533",99.43%
listings_residential,CoBuyerAgentFirstName,more than 90% missing,"601,575",97.65%


### Data Types and Missing Values

Dates were parsed and saved as `YYYY-MM-DD`. Key prices, property fields,
coordinates, timing fields, and mortgage rates were converted to numeric
values. No nonblank values failed conversion.

In [9]:
conversion_summary = (
    data_types.groupby("dataset")
    .agg(
        fields_confirmed=("column", "count"),
        conversion_failures=(
            "nonblank_values_that_failed_conversion",
            "sum",
        ),
    )
    .reset_index()
)
conversion_summary.columns = [
    "Dataset",
    "Date and Numeric Fields Confirmed",
    "Nonblank Conversion Failures",
]

display(
    conversion_summary.style
    .hide(axis="index")
    .format(
        {
            "Date and Numeric Fields Confirmed": "{:,.0f}",
            "Nonblank Conversion Failures": "{:,.0f}",
        }
    )
)

complete_types_html = data_types.to_html(index=False, border=0)
display(
    HTML(
        "<details><summary><b>Open complete data-type report</b></summary>"
        "<div style='overflow-x:auto;max-height:500px;overflow-y:auto;"
        "margin-top:10px;'>"
        + complete_types_html
        + "</div></details>"
    )
)

Dataset,Date and Numeric Fields Confirmed,Nonblank Conversion Failures
listings_residential,17,0
sold_residential,17,0


dataset,column,cleaning_type,csv_format,nonblank_values_that_failed_conversion
listings_residential,BathroomsTotalInteger,float64,numeric,0
listings_residential,BedroomsTotal,float64,numeric,0
listings_residential,CloseDate,datetime64[ns],YYYY-MM-DD,0
listings_residential,ClosePrice,float64,numeric,0
listings_residential,ContractStatusChangeDate,datetime64[ns],YYYY-MM-DD,0
listings_residential,DaysOnMarket,float64,numeric,0
listings_residential,Latitude,float64,numeric,0
listings_residential,ListPrice,float64,numeric,0
listings_residential,ListingContractDate,datetime64[ns],YYYY-MM-DD,0
listings_residential,LivingArea,float64,numeric,0


## Week 5: California-Only Filter

Coordinates are the primary location check. Rows with missing coordinates are
kept only when the original state is `CA`. Rows whose coordinates confirm
California but whose state label is wrong are corrected to `CA`; the original
label is retained in `StateOrProvinceOriginal`.

In [10]:
california_flow = pd.DataFrame(
    [
        {
            "Dataset": "Residential Listings",
            "Starting Rows": cleaning_value("listings_residential", "source_rows"),
            "Kept by Coordinates": cleaning_value("listings_residential", "rows_inside_california_boundary"),
            "Kept by CA State Fallback": cleaning_value("listings_residential", "rows_kept_by_ca_state_fallback"),
            "Removed Outside CA": cleaning_value("listings_residential", "rows_removed_outside_california"),
            "Removed Missing Coordinates/Non-CA": cleaning_value("listings_residential", "rows_removed_missing_coordinates_non_ca_state"),
            "State Labels Corrected": cleaning_value("listings_residential", "state_labels_corrected_from_coordinates"),
            "Final Rows": cleaning_value("listings_residential", "output_rows"),
        },
        {
            "Dataset": "Residential Sold",
            "Starting Rows": cleaning_value("sold_residential", "source_rows"),
            "Kept by Coordinates": cleaning_value("sold_residential", "rows_inside_california_boundary"),
            "Kept by CA State Fallback": cleaning_value("sold_residential", "rows_kept_by_ca_state_fallback"),
            "Removed Outside CA": cleaning_value("sold_residential", "rows_removed_outside_california"),
            "Removed Missing Coordinates/Non-CA": cleaning_value("sold_residential", "rows_removed_missing_coordinates_non_ca_state"),
            "State Labels Corrected": cleaning_value("sold_residential", "state_labels_corrected_from_coordinates"),
            "Final Rows": cleaning_value("sold_residential", "output_rows"),
        },
    ]
)

display(
    california_flow.style
    .hide(axis="index")
    .format(
        {
            column: "{:,.0f}"
            for column in california_flow.columns
            if column != "Dataset"
        }
    )
)

Dataset,Starting Rows,Kept by Coordinates,Kept by CA State Fallback,Removed Outside CA,Removed Missing Coordinates/Non-CA,State Labels Corrected,Final Rows
Residential Listings,"616,072","534,696","80,983",365,28,16,"615,679"
Residential Sold,"447,964","443,479","4,374",111,0,7,"447,853"


In [11]:
removed_states = cleaning[
    cleaning["category"].eq("removed_state_label")
][["dataset", "metric", "value"]].copy()
removed_states.columns = ["Dataset", "Original State Label", "Rows Removed"]
removed_states = removed_states.sort_values(
    ["Dataset", "Rows Removed"],
    ascending=[True, False],
)

display(HTML("<h3>Original state labels among removed rows</h3>"))
display(
    removed_states.style
    .hide(axis="index")
    .format({"Rows Removed": "{:,.0f}"})
)

Dataset,Original State Label,Rows Removed
listings_residential,CA,184
listings_residential,OS,82
listings_residential,MISSING,62
listings_residential,AZ,14
listings_residential,OR,7
listings_residential,FL,5
listings_residential,NV,4
listings_residential,TX,4
listings_residential,BC,3
listings_residential,CO,3


### Geographic Data Quality

Zero coordinates and positive longitudes are invalid for California and were
excluded by the coordinate boundary. Missing-coordinate rows retained through
the CA-state fallback remain flagged in the final datasets.

In [12]:
geographic_metrics = {
    "rows_missing_coordinates": "Rows Missing Coordinates",
    "zero_coordinate_rows": "Zero-Coordinate Rows",
    "positive_longitude_rows": "Positive-Longitude Rows",
}
geographic_rows = []
for dataset_name, dataset_label in [
    ("listings_residential", "Residential Listings"),
    ("sold_residential", "Residential Sold"),
]:
    for metric, label in geographic_metrics.items():
        geographic_rows.append(
            {
                "Dataset": dataset_label,
                "Check": label,
                "Rows": cleaning_value(dataset_name, metric),
            }
        )

geographic_review = pd.DataFrame(geographic_rows)
display(
    geographic_review.style
    .hide(axis="index")
    .format({"Rows": "{:,.0f}"})
)

Dataset,Check,Rows
Residential Listings,Rows Missing Coordinates,"81,011"
Residential Listings,Zero-Coordinate Rows,75
Residential Listings,Positive-Longitude Rows,85
Residential Sold,Rows Missing Coordinates,"4,374"
Residential Sold,Zero-Coordinate Rows,37
Residential Sold,Positive-Longitude Rows,31


### Date and Timeline Inconsistencies

Timeline issues are flagged rather than automatically deleted because the row
may still contain useful price, location, or competitive-analysis information.

In [13]:
cleaned_files = {
    "Residential Listings": (
        "listings_residential",
        CLEANED_DIR / "listings_residential_california_clean.csv",
    ),
    "Residential Sold": (
        "sold_residential",
        CLEANED_DIR / "sold_residential_california_clean.csv",
    ),
}
timeline_rows = []

for dataset_label, (dataset_name, cleaned_path) in cleaned_files.items():
    dates = pd.read_csv(
        cleaned_path,
        usecols=["ListingContractDate", "PurchaseContractDate", "CloseDate"],
    )
    for field in dates.columns:
        dates[field] = pd.to_datetime(dates[field], errors="coerce")
    listing_after_purchase = (
        dates["ListingContractDate"] > dates["PurchaseContractDate"]
    ).sum()

    timeline_rows.extend(
        [
            {
                "Dataset": dataset_label,
                "Inconsistency": "Listing date after close date",
                "Flagged Rows": cleaning_value(dataset_name, "listing_after_close_flag"),
            },
            {
                "Dataset": dataset_label,
                "Inconsistency": "Purchase date after close date",
                "Flagged Rows": cleaning_value(dataset_name, "purchase_after_close_flag"),
            },
            {
                "Dataset": dataset_label,
                "Inconsistency": "Listing date after purchase date",
                "Flagged Rows": int(listing_after_purchase),
            },
            {
                "Dataset": dataset_label,
                "Inconsistency": "Any out-of-order timeline",
                "Flagged Rows": cleaning_value(dataset_name, "negative_timeline_flag"),
            },
        ]
    )

timeline_review = pd.DataFrame(timeline_rows)
timeline_review["Treatment"] = "Flagged; row retained"
display(
    timeline_review.style
    .hide(axis="index")
    .format({"Flagged Rows": "{:,.0f}"})
)

Dataset,Inconsistency,Flagged Rows,Treatment
Residential Listings,Listing date after close date,84,Flagged; row retained
Residential Listings,Purchase date after close date,268,Flagged; row retained
Residential Listings,Listing date after purchase date,301,Flagged; row retained
Residential Listings,Any out-of-order timeline,568,Flagged; row retained
Residential Sold,Listing date after close date,67,Flagged; row retained
Residential Sold,Purchase date after close date,239,Flagged; row retained
Residential Sold,Listing date after purchase date,290,Flagged; row retained
Residential Sold,Any out-of-order timeline,529,Flagged; row retained


## Final Outputs

- `listings_residential_california_clean.csv`
- `sold_residential_california_clean.csv`
- `cleaning_summary.csv`
- `column_removal_report.csv`
- `data_type_report.csv`

The cleaned datasets are ready for the next analysis and Tableau preparation
steps.

## Sources

- CRMLS monthly listing and sold files, January 2024 through June 2026
- Mortgage-enriched Residential listing and sold datasets
- Week 4-5 cleaning summaries and California-only output datasets